# 3.2 權重能由內容決定嗎？


查資料時，問「哪條是紅色物體？」和問「哪條是圓形物體？」，應該關注不同記錄。[3.1](https://birdhackor.github.io/tiny-perceptron-vlm/3.1.html) 手動給讀取比例，現在讓一個表示需求的向量與各位置的線索比較，匹配程度高的就取得較多比例。模型就不用為所有問題永遠採相同平均。

為了能逐數追蹤，先用兩特徵的小世界。當前需求寫成q=`[1,0]`，三個候選線索分別 `[1,0]`、`[0,1]`、`[-1,0]`。我們把需求向量叫query（查詢），候選的匹配線索叫key（鍵）。這是臨時人為指定的數值，不是說真實模型第一格一定代表顏色；模型會在後續學習怎樣形成有用的線索。

如何匹配？對應格相乘，再相加，這叫點積（dot product），與 [W.5](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.5) 一套加權配方相同。q與第一候選得到 `1×1+0×0=1`，與第二得到0，第三得到-1。然後按 [1.7](https://birdhackor.github.io/tiny-perceptron-vlm/1.7.html) 的softmax換成非負、加總為1的讀取權重。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch

q = torch.tensor([1.0, 0.0])
keys = torch.tensor([[1.0, 0.0], [0.0, 1.0], [-1.0, 0.0]])
scores = q @ keys.T
weights = scores.softmax(dim=0)
print(scores)
print(weights)

輸入是一個需求和三個候選，各有兩數。`keys.T` 交換列欄，讓 `[2] @ [2,3]` 產生三個分數。輸出先是 `[1,0,-1]`，再約為 `[0.665,0.245,0.090]`。第一候選最多，但其餘仍參與。把需求q改成 `[0,1]`，分數變成 `[0,1,0]`，第二候選才會得到最多比例，說明讀取偏好由當前內容決定。

點積常被解釋成方向是否相近，但也受長度影響。把第一key從 `[1,0]` 改成 `[2,0]`，方向完全沒變，對原q的分數卻從1升2。不要把它當作已經排除長度差的相似度。這種分數在訓練中可以被利用，但過大時會讓softmax過於集中，稍後 [3.5](https://birdhackor.github.io/tiny-perceptron-vlm/3.5.html) 會處理尺度。

這段還沒有取回資料，也沒有限制能讀哪些位置，只完成匹配與比例化。若用在猜下一字，不能讓查詢去比對未來字的key，否則會偷看到答案；限制會在 [3.6](https://birdhackor.github.io/tiny-perceptron-vlm/3.6.html) 另做。本節先讓三候選都可見，清楚觀察需求變化。

練習把q的兩個數改成 `[0,1]`，先寫下三個分數及哪個候選最高，再執行核對：應為 `[0,1,0]`，第二個候選得到最多比例。保持這個q，再把keys最後一列`[-1,0]`改成`[-1,1]`，只動第二格。先手算第三個候選也得1，預測後兩個候選並列最高、權重相等，再執行看它們各佔約0.422。需求與候選都影響讀法，沒有固定哪個位置永遠重要。
